In [1]:
FRAMEWORK = 'modin'

# Proyecto Big Data - Modin

## 0. Instalación, entorno y acceso a los datos

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q modin[ray] ray pyarrow pandas kagglehub
print('Entorno:', 'Google Colab' if IN_COLAB else 'local')

Entorno: local


In [3]:
from pathlib import Path
import shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/BigD')
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / 'data').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'bank_transactions.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / FRAMEWORK
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    import kagglehub
    downloaded = Path(kagglehub.dataset_download('shivamb/bank-customer-segmentation'))
    shutil.copy2(downloaded / 'bank_transactions.csv', DATA_PATH)

print(f'Dataset: {DATA_PATH}')
print(f'Resultados: {OUTPUT_DIR}')

Dataset: /Users/alejandromarcelo/Documents/ChatGPT/BigD/data/raw/bank_transactions.csv
Resultados: /Users/alejandromarcelo/Documents/ChatGPT/BigD/outputs/modin


## 1. Carga e inspección con Modin y Ray

In [4]:
import os
import math
import warnings
os.environ['MODIN_ENGINE'] = 'ray'

import modin
import modin.pandas as mpd
import ray

FALLBACKS = []
def registrar_fallback(message, category, filename, lineno, file=None, line=None):
    if 'pandas' in str(message).lower():
        FALLBACKS.append(str(message).splitlines()[0][:160])
warnings.simplefilter('always', UserWarning)
warnings.showwarning = registrar_fallback

raw = mpd.read_csv(DATA_PATH, keep_default_na=False, na_values=['', 'nan'])
print('Motor:', os.environ['MODIN_ENGINE'])
print('Versión Modin:', modin.__version__, '| Ray:', ray.__version__)
print('Recursos Ray:', ray.cluster_resources())
print('Dimensiones:', raw.shape)
display(raw.head())
raw.info()

2026-09-16 09:19:13,007	INFO worker.py:2024 -- Started a local Ray instance.


Motor: ray
Versión Modin: 0.37.1 | Ray: 2.58.0
Recursos Ray: {'memory': 2147483648.0, 'object_store_memory': 2147483648.0, 'GPU': 1.0, 'node:__internal_head__': 1.0, 'CPU': 10.0, 'accelerator_type:M2-Pro': 1.0, 'node:127.0.0.1': 1.0}
Dimensiones: (1048567, 9)


,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR)
0,T1,C5841053,10/1/94,F,JAMSHEDPUR,17819.05,2/8/16,143207,25.0
1,T2,C2142763,4/4/57,M,JHAJJAR,2270.69,2/8/16,141858,27999.0
2,T3,C4417068,26/11/96,F,MUMBAI,17874.44,2/8/16,142712,459.0
3,T4,C5342380,14/9/73,F,MUMBAI,866503.21,2/8/16,142714,2060.0
4,T5,C9031234,24/3/88,F,NAVI MUMBAI,6714.43,2/8/16,181156,1762.5


<class 'modin.pandas.dataframe.DataFrame'>
RangeIndex: 1048567 entries, 0 to 1048566
Data columns (total 9 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   TransactionID            1048567 non-null  object 
 1   CustomerID               1048567 non-null  object 
 2   CustomerDOB              1045170 non-null  object 
 3   CustGender               1047467 non-null  object 
 4   CustLocation             1048416 non-null  object 
 5   CustAccountBalance       1046198 non-null  float64
 6   TransactionDate          1048567 non-null  object 
 7   TransactionTime          1048567 non-null  int64  
 8   TransactionAmount (INR)  1048567 non-null  float64
dtypes: float64(2), int64(1), object(6)
memory usage: 72.0+ MB


## 2. Preparación común

In [5]:
AMOUNT = 'TransactionAmount (INR)'
BALANCE = 'CustAccountBalance'

transaction_date = mpd.to_datetime(
    raw['TransactionDate'], format='%d/%m/%y', errors='coerce'
)
txn_year = transaction_date.dt.year

dob_parts = raw['CustomerDOB'].str.split('/')
dob_day = mpd.to_numeric(dob_parts.str[0], errors='coerce')
dob_month = mpd.to_numeric(dob_parts.str[1], errors='coerce')
dob_year_raw = mpd.to_numeric(dob_parts.str[2], errors='coerce')

short_year = (dob_year_raw < 100).fillna(False).astype('int64')
current_century = (
    (dob_year_raw < 100) & (dob_year_raw <= (txn_year % 100))
).fillna(False).astype('int64')
birth_year = dob_year_raw + short_year * (1900 + 100 * current_century)

birthday_not_reached = (
    (transaction_date.dt.month < dob_month)
    | (
        (transaction_date.dt.month == dob_month)
        & (transaction_date.dt.day < dob_day)
    )
).fillna(False).astype('int64')
age = txn_year - birth_year - birthday_not_reached
age[~age.between(18, 100)] = float('nan')

def rango_etario(edad):
    if edad != edad:
        return 'Desconocido'
    if edad <= 25:
        return '18-25'
    if edad <= 35:
        return '26-35'
    if edad <= 45:
        return '36-45'
    if edad <= 60:
        return '46-60'
    return '61-100'

hour = mpd.to_numeric(raw['TransactionTime'], errors='coerce') // 10000
FRANJAS = {0: 'Madrugada', 1: 'Manana', 2: 'Tarde', 3: 'Noche'}

base = raw.copy()
base['TransactionDateParsed'] = transaction_date
base['LocationNormalized'] = (
    raw['CustLocation'].astype('string').str.strip().str.upper()
)
base['Age'] = age
base['Hour'] = hour
base['TimeBand'] = (hour // 6).map(FRANJAS).fillna('Invalida')
base['AgeRange'] = age.map(rango_etario)

clean = base.drop_duplicates(['TransactionID']).drop_duplicates(
    ['CustomerID', 'TransactionDate', 'TransactionTime', AMOUNT]
)
print('Filas preparadas:', len(clean))
print('Fallbacks a pandas hasta ahora:', FALLBACKS or 'ninguno')

Filas preparadas: 1048567
Fallbacks a pandas hasta ahora: ninguno


## Pregunta 1. Diagnóstico de calidad de datos

In [6]:
q01 = mpd.DataFrame({
    'column': raw.columns,
    'null_count': raw.isna().sum().values,
})
q01['null_percent'] = (q01['null_count'] * 100 / len(raw)).round(4)
q01.to_csv(OUTPUT_DIR / 'q01_calidad_datos.csv', index=False)
display(q01)

,column,null_count,null_percent
0,TransactionID,0,0.0000
1,CustomerID,0,0.0000
2,CustomerDOB,3397,0.3240
3,CustGender,1100,0.1049
4,CustLocation,151,0.0144
5,CustAccountBalance,2369,0.2259
6,TransactionDate,0,0.0000
7,TransactionTime,0,0.0000
8,TransactionAmount (INR),0,0.0000


## Pregunta 2. Detección y tratamiento de duplicados

In [7]:
by_id = int(raw.duplicated(['TransactionID']).sum())
by_combo = int(raw.duplicated(['CustomerID', 'TransactionDate', AMOUNT]).sum())
by_event = int(raw.duplicated(
    ['CustomerID', 'TransactionDate', 'TransactionTime', AMOUNT]
).sum())
q02 = mpd.DataFrame({
    'criterion': [
        'TransactionID',
        'CustomerID+TransactionDate+Amount',
        'CustomerID+TransactionDate+Time+Amount',
    ],
    'duplicate_rows': [by_id, by_combo, by_event],
    'treatment': [
        'Eliminar repetidos',
        'Conservar y revisar: ocurren a horas distintas',
        'Eliminar repetidos exactos del evento',
    ],
})
q02.to_csv(OUTPUT_DIR / 'q02_duplicados.csv', index=False)
display(q02)

,criterion,duplicate_rows,treatment
0,TransactionID,0,Eliminar repetidos
1,CustomerID+TransactionDate+Amount,31,Conservar y revisar: ocurren a horas distintas
2,CustomerID+TransactionDate+Time+Amount,0,Eliminar repetidos exactos del evento


## Pregunta 3. Edad exacta y rangos etarios

In [8]:
q03 = clean.groupby('AgeRange', dropna=False, observed=True).agg(
    transactions=('TransactionID', 'size'),
    mean_age=('Age', 'mean'),
).reset_index().sort_values('AgeRange')
q03['mean_age'] = q03['mean_age'].round(2)
q03.to_csv(OUTPUT_DIR / 'q03_edades.csv', index=False)
display(q03)

,AgeRange,transactions,mean_age
0,18-25,295117,23.08
1,26-35,482990,29.62
2,36-45,142585,39.43
3,46-60,50856,51.03
4,61-100,14303,67.10
5,Desconocido,62716,NaN


## Pregunta 4. Franja horaria y ubicación normalizada

In [9]:
q04 = clean.groupby('TimeBand', dropna=False).agg(
    transactions=('TransactionID', 'size'),
    unique_locations=('LocationNormalized', 'nunique'),
).reset_index().sort_values(['transactions', 'TimeBand'], ascending=[False, True])
q04.to_csv(OUTPUT_DIR / 'q04_franja_ubicacion.csv', index=False)
display(q04)

,TimeBand,transactions,unique_locations
2,Noche,436179,7246
3,Tarde,390884,7146
1,Manana,174473,4889
0,Madrugada,47031,2208


## Pregunta 5. Outliers por percentiles 1 y 99

In [10]:
balance_p01, balance_p99 = clean[BALANCE].quantile([0.01, 0.99]).values
amount_p01, amount_p99 = clean[AMOUNT].quantile([0.01, 0.99]).values
q05 = mpd.DataFrame({
    'variable': [BALANCE, AMOUNT],
    'p01': [balance_p01, amount_p01],
    'p99': [balance_p99, amount_p99],
    'outlier_rows': [
        int(((clean[BALANCE] < balance_p01) | (clean[BALANCE] > balance_p99)).sum()),
        int(((clean[AMOUNT] < amount_p01) | (clean[AMOUNT] > amount_p99)).sum()),
    ],
})
q05.to_csv(OUTPUT_DIR / 'q05_outliers.csv', index=False)
display(q05)

,variable,p01,p99,outlier_rows
0,CustAccountBalance,3.23,1586901.17,20913
1,TransactionAmount (INR),8.00,20000.00,20407


## Pregunta 6. Balance y monto promedio por género y edad

In [11]:
valid = clean[clean['CustGender'].notna() & clean['Age'].notna()]
q06 = valid.groupby(['CustGender', 'AgeRange'], observed=True).agg(
    transactions=('TransactionID', 'size'),
    avg_balance=(BALANCE, 'mean'),
    avg_amount=(AMOUNT, 'mean'),
).reset_index().sort_values(['CustGender', 'AgeRange'])
q06[['avg_balance', 'avg_amount']] = q06[['avg_balance', 'avg_amount']].round(2)
q06.to_csv(OUTPUT_DIR / 'q06_genero_edad.csv', index=False)
display(q06)

,CustGender,AgeRange,transactions,avg_balance,avg_amount
0,F,18-25,94727,37858.98,1007.18
1,F,26-35,125778,84128.28,1620.86
2,F,36-45,34248,200026.15,2323.98
3,F,46-60,14126,267580.73,3171.41
4,F,61-100,4210,695495.59,3080.99
5,M,18-25,200390,33768.21,803.19
6,M,26-35,357212,84804.58,1275.34
7,M,36-45,108337,190350.65,2155.85
8,M,46-60,36730,348567.61,2973.25
9,M,61-100,9942,649675.48,3703.29


## Pregunta 7. Top 20 ciudades

In [12]:
q07 = clean[clean['LocationNormalized'].notna()].groupby('LocationNormalized').agg(
    transactions=('TransactionID', 'size'),
    total_amount=(AMOUNT, 'sum'),
).reset_index().sort_values(
    ['total_amount', 'LocationNormalized'], ascending=[False, True]
).head(20)
q07['total_amount'] = q07['total_amount'].round(2)
q07.to_csv(OUTPUT_DIR / 'q07_top_ciudades.csv', index=False)
display(q07)

,LocationNormalized,transactions,total_amount
5267,MUMBAI,103596,1.796891e+08
5790,NEW DELHI,84928,1.607059e+08
772,BANGALORE,81555,1.184248e+08
3083,GURGAON,73818,1.120947e+08
2075,DELHI,71019,1.062249e+08
4267,KOLKATA,19974,6.060031e+07
1603,CHENNAI,30009,4.463782e+07
5886,NOIDA,32784,4.446343e+07
6717,PUNE,25851,3.959035e+07
3394,HYDERABAD,23049,3.617739e+07


## Pregunta 8. Cliente con mayor gasto por ciudad

In [13]:
spending = clean[
    clean['LocationNormalized'].notna() & clean['CustomerID'].notna()
].groupby(['LocationNormalized', 'CustomerID']).agg(
    total_spent=(AMOUNT, 'sum'),
    transactions=('TransactionID', 'size'),
).reset_index()
q08 = spending.sort_values(
    ['LocationNormalized', 'total_spent', 'CustomerID'],
    ascending=[True, False, True],
).drop_duplicates('LocationNormalized').sort_values(
    ['total_spent', 'LocationNormalized'], ascending=[False, True]
)
q08['city_rank'] = 1
q08['total_spent'] = q08['total_spent'].round(2)
q08.to_csv(OUTPUT_DIR / 'q08_top_cliente_ciudad.csv', index=False)
print('Ciudades:', len(q08))
display(q08.head(20))

Ciudades: 9353


,LocationNormalized,CustomerID,total_spent,transactions,city_rank
414366,GURGAON,C7319271,1560034.99,1,1
898233,PUNE,C6677159,1380002.88,1,1
766942,NEW DELHI,C4141768,991132.22,1,1
688190,MUMBAI,C8217728,724122.00,1,1
529169,KOLKATA,C1830891,720001.16,1,1
842057,NOIDA,C6549785,600008.32,1,1
252369,DELHI,C4328064,569500.27,1,1
857155,PALAKKARAI TRICHY,C5833636,557000.73,1,1
569623,LUDHIANA,C8755262,514320.00,1,1
96801,BANGALORE,C5720892,500000.00,1,1


## Pregunta 9. Serie temporal diaria

In [14]:
q09 = clean[clean['TransactionDateParsed'].notna()].groupby(
    'TransactionDateParsed'
).agg(
    transactions=('TransactionID', 'size'),
    total_amount=(AMOUNT, 'sum'),
).reset_index().sort_values('TransactionDateParsed')
q09['total_amount'] = q09['total_amount'].round(2)
q09.to_csv(OUTPUT_DIR / 'q09_serie_diaria.csv', index=False)
display(q09)

,TransactionDateParsed,transactions,total_amount
0,2016-08-01,20438,29801816.34
1,2016-08-02,20948,30467503.29
2,2016-08-03,20615,31149483.67
3,2016-08-04,20682,35722718.64
4,2016-08-05,21112,34833933.12
5,2016-08-06,26585,47527227.82
6,2016-08-07,27261,45727772.63
7,2016-08-08,21042,30129433.64
8,2016-08-09,21823,33479570.48
9,2016-08-10,21649,32016308.51


## Pregunta 10. Ratio gasto/balance y top 1%

In [15]:
valid = clean[
    clean[BALANCE].notna() & (clean[BALANCE] > 0) & clean[AMOUNT].notna()
].copy()
valid['spend_balance_ratio'] = valid[AMOUNT] / valid[BALANCE]
top_count = math.ceil(len(valid) * 0.01)
q10 = valid.sort_values(
    ['spend_balance_ratio', 'TransactionID'], ascending=[False, True]
).head(top_count)[[
    'TransactionID', 'CustomerID', 'LocationNormalized',
    BALANCE, AMOUNT, 'spend_balance_ratio'
]]
q10.to_csv(OUTPUT_DIR / 'q10_ratio_top1.csv', index=False)
print('Filas válidas:', len(valid), '| Filas del top 1%:', len(q10))
display(q10.head(20))

Filas válidas: 1043487 | Filas del top 1%: 10435


,TransactionID,CustomerID,LocationNormalized,CustAccountBalance,TransactionAmount (INR),spend_balance_ratio
742110,T742111,C7323566,CHANDIGARH,0.01,42398.0,4239800.0
836116,T836117,C5719489,KOLAR,0.01,25500.0,2550000.0
253452,T253453,C3523520,CHANDIGARH,0.01,20000.0,2000000.0
652734,T652735,C6038911,TANK HYDERABAD,0.01,17820.0,1782000.0
421849,T421850,C2816416,MUMBAI,0.01,15715.0,1571500.0
424044,T424045,C7338987,TANK HYDERABAD,0.01,10764.0,1076400.0
343015,T343016,C2223587,CHANDIGARH,0.01,9500.0,950000.0
782381,T782382,C8238539,LUCKNOW,0.01,9000.0,900000.0
311364,T311365,C5527984,FARIDABAD,0.01,8700.0,870000.0
196758,T196759,C1920771,CHENNAI,0.05,42625.0,852500.0


In [16]:
print('Fallbacks a pandas detectados:', sorted(set(FALLBACKS)) or 'ninguno')

Fallbacks a pandas detectados: ninguno
